# 14 · Does the 67% -> 99% difference survive a second training seed?

Every configuration so far was trained once (seed 0). This notebook retrains the two datasets of the key comparison, `ours_consistent_clean` (my approach) and `ours_consistent_vertical` (planned approach), with training seed 1: same data (pulled from the Hub, no generation), same settings, two evaluations each. Run once per `VARIANT`, in two runtimes in parallel.

**Predictions, fixed before running:** the comparison replicates if `vertical` (seed 1) >= 90% and `clean` (seed 1) <= 75%. Either side outside these bounds means the seed-0 numbers depended on the seed more than their intervals suggest.

**Runtime:** T4, about 1.7 h per variant.

In [ ]:
import os, sys, json, subprocess
os.environ["MUJOCO_GL"] = "egl"                      # GPU rendering, must be set before mujoco is imported
REPO_URL = "https://github.com/LukasWill/phone2panda.git"
BASE = "/content" if os.path.exists("/content") else "/kaggle/working"   # Colab or Kaggle
CODE_DIR = f"{BASE}/phone2panda"
if not os.path.exists(CODE_DIR):
    subprocess.run(["git", "clone", "-q", REPO_URL, CODE_DIR], check=True)
else:
    subprocess.run(["git", "-C", CODE_DIR, "pull", "-q"], check=True)
# LIBERO needs egl_probe / hf-egl-probe, which are built from source with cmake. pip's isolated build cannot run
# the pip-installed cmake wrapper, and the old egl_probe CMakeLists is rejected by cmake 4: build both first,
# without isolation and with the CMake 3.5 policy floor.
os.environ["CMAKE_POLICY_VERSION_MINIMUM"] = "3.5"
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "setuptools", "wheel", "cmake>=3.29,<4.2"], check=True)
subprocess.run([sys.executable, "-m", "pip", "install", "-q", "--no-build-isolation", "egl_probe", "hf-egl-probe"], check=True)
r = subprocess.run([sys.executable, "-m", "pip", "install", "-q", "lerobot[libero]==0.6.1", "imageio-ffmpeg"],
                   capture_output=True, text=True)
if r.returncode != 0:                                  # show pip's own reason, not just "exit status 1"
    print((r.stdout + r.stderr)[-4000:]); raise RuntimeError("pip install failed: see the lines above")
# Colab preinstalls an older torchao; a newer diffusers (pulled in by lerobot) then fails on import
# (FqnToConfig). ACT does not use torchao, and diffusers skips it when it is absent.
subprocess.run([sys.executable, "-m", "pip", "uninstall", "-y", "-q", "torchao"], check=False)
os.chdir(CODE_DIR); sys.path.insert(0, CODE_DIR)
from p2p.sim.env import ensure_nvidia_egl
ensure_nvidia_egl()      # Colab: make EGL render on the GPU (otherwise it silently renders on the CPU, ~10x slower)
# Hugging Face token (Colab: key icon -> Secrets -> HF_TOKEN; Kaggle: Add-ons -> Secrets -> HF_TOKEN)
try:
    from google.colab import userdata; os.environ["HF_TOKEN"] = userdata.get("HF_TOKEN")
except Exception:
    try:
        from kaggle_secrets import UserSecretsClient; os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        print("No HF_TOKEN found: uploads to the Hub will fail")
if os.environ.get("HF_TOKEN"):
    from huggingface_hub import whoami
    print("Hugging Face account:", whoami(token=os.environ["HF_TOKEN"])["name"])
import torch; print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "none")

In [ ]:
VARIANT = "vertical"       # "vertical" or "clean": run this notebook twice, once with each
HUB_REPO, SEED, STEPS, BATCH = "phone2panda-data", 1, 20000, 32
NAME = f"ours_consistent_{VARIANT}"
RUN = f"act_{NAME}_s{SEED}"
from p2p.hub import pull
pull(HUB_REPO, f"{BASE}/hub", allow_patterns=[f"lerobot/{NAME}/*"])

In [ ]:
!python -m p2p.policy.act train {BASE}/hub/lerobot/{NAME} {BASE}/runs/{RUN} --steps {STEPS} --batch-size {BATCH} --seed {SEED}

In [ ]:
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 6 --out {BASE}/runs/{RUN}/eval
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --out {BASE}/runs/{RUN}/eval_rerun
!python -m p2p.policy.act eval {BASE}/runs/{RUN}/final --episodes 50 --videos 0 --temporal-ensemble 0.01 --out {BASE}/runs/{RUN}/eval_te
import glob
evals = " ".join(sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")))
!python tools/policy_failures.py {evals}
for e in sorted(glob.glob(f"{BASE}/runs/{RUN}/eval*/eval.json")):
    print(e.split("/")[-2], json.dumps(json.load(open(e))["behaviour"]))

In [ ]:
from p2p.hub import push
push(f"{BASE}/runs/{RUN}", HUB_REPO, path_in_repo=f"runs/{RUN}")
!cd {BASE}/runs && zip -qr {BASE}/{RUN}.zip {RUN} -x "*/final/*" "*/step_*"
try:
    from google.colab import files; files.download(f"{BASE}/{RUN}.zip")
except Exception:
    print("download", f"{BASE}/{RUN}.zip", "from the Output panel")